[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/04_tag_11_12_rnn_hidden_state.ipynb)

# Tag 11 & 12 – 04: Mit einem RNN den nächsten Zeitreihenwert vorhersagen

Ein RNN liest die Werte eines Zeitfensters nacheinander und fasst die bisherige Information im **Hidden State** zusammen. Nach dem letzten Wert eines Fensters erzeugt ein Dense-Layer eine Prognose für den unmittelbar folgenden Zeitreihenwert.

## Vorhersageaufgabe

Die künstliche Zeitreihe kombiniert einen langsamen und einen schnelleren periodischen Verlauf mit leichtem Messrauschen. Aus jeweils 20 aufeinanderfolgenden Messwerten soll der 21. Wert vorhergesagt werden.

$$[x_{t-19}, \ldots, x_t] \; \longrightarrow \; x_{t+1}$$

Der Trainingsbereich liegt vollständig vor dem Testbereich. Das RNN sieht im Training nur frühere Zeitabschnitte und wird anschließend auf späteren Zielwerten bewertet.

In [ ]:
import matplotlib.pyplot as plt  # Grafiken für Zeitreihe, Loss und Hidden States
import numpy as np               # Arrays und mathematische Berechnungen
import tensorflow as tf          # RNN-Modell und Training

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42                        # Gleiche Zufallswerte bei jedem Notebook-Lauf
np.random.seed(RANDOM_STATE)              # NumPy-Zufallsgenerator festlegen
tf.keras.utils.set_random_seed(RANDOM_STATE)  # TensorFlow-Gewichte reproduzierbar initialisieren

## Zeitreihe vorbereiten

Für die Skalierung werden ausschließlich Werte aus dem Trainingsabschnitt verwendet. Anschließend entstehen überlappende 20er-Fenster. Bei einem Testfenster dürfen die letzten bekannten Trainingswerte als Vorgeschichte enthalten sein; der vorherzusagende Zielwert liegt jedoch immer im Testbereich.

In [ ]:
ANZAHL_WERTE = 360       # Länge der gesamten künstlichen Zeitreihe
FENSTERLAENGE = 20       # Anzahl vergangener Werte pro Ein-Schritt-Prognose
TRAINING_ANTEIL = 0.70   # Früher Abschnitt zum Lernen, späterer Abschnitt zum Testen

zeit = np.arange(ANZAHL_WERTE)             # Zeitindizes 0, 1, ..., 359
rng = np.random.default_rng(RANDOM_STATE)  # Lokaler Zufallsgenerator
werte = (  # Beide periodischen Komponenten und Messrauschen zu einer Zeitreihe addieren
    1.00 * np.sin(2 * np.pi * zeit / 30)  # Langsame Schwingung mit Periode 30
    + 0.35 * np.sin(2 * np.pi * zeit / 10 + 0.7)  # Schnellere Zusatzschwingung mit Periode 10
    + rng.normal(0, 0.08, ANZAHL_WERTE)  # Kleines unabhängiges Messrauschen
)
trennindex = int(ANZAHL_WERTE * TRAINING_ANTEIL)  # Erste Position des Testbereichs

# Skalierung wird ausschließlich aus Trainingswerten berechnet.
mittelwert = werte[:trennindex].mean()
standardabweichung = werte[:trennindex].std()
werte_skaliert = (werte - mittelwert) / standardabweichung

def erstelle_fenster(reihe, fensterlaenge):
    X, y, zielindizes = [], [], []  # Listen für Fenster, Zielwerte und ihre Positionen
    for start in range(len(reihe) - fensterlaenge):
        X.append(reihe[start : start + fensterlaenge])  # Vergangene Werte als Eingabe
        y.append(reihe[start + fensterlaenge])           # Unmittelbar nächster Wert als Ziel
        zielindizes.append(start + fensterlaenge)         # Originalposition des Zielwerts
    return np.asarray(X)[..., np.newaxis], np.asarray(y), np.asarray(zielindizes)  # (Beispiele, Zeit, Feature) zurückgeben


X_alle, y_alle, zielindizes = erstelle_fenster(werte_skaliert, FENSTERLAENGE)
train_maske = zielindizes < trennindex   # Ziele vor der Grenze gehören zum Training
test_maske = ~train_maske                # Ziele ab der Grenze gehören zum Test
X_train, y_train = X_alle[train_maske], y_alle[train_maske]  # Frühere Fenster und Ziele auswählen
X_test, y_test = X_alle[test_maske], y_alle[test_maske]      # Spätere Fenster und Ziele auswählen
test_indizes = zielindizes[test_maske]                        # Zeitpositionen der Testziele merken

print('X_train:', X_train.shape, '| y_train:', y_train.shape)  # Formen der Trainingsdaten ausgeben
print('X_test: ', X_test.shape, '| y_test: ', y_test.shape)    # Formen der Testdaten ausgeben

plt.figure(figsize=(12, 3.8))  # Neue Abbildung mit breitem Zeitachsenformat anlegen
plt.plot(zeit, werte, color='0.35', linewidth=1.3, label='Zeitreihe')  # Gesamten Signalverlauf zeichnen
plt.axvspan(0, trennindex - 1, color='tab:blue', alpha=0.10, label='Training')  # Trainingsbereich einfärben
plt.axvspan(trennindex, ANZAHL_WERTE - 1, color='tab:orange', alpha=0.10, label='Test')  # Testbereich einfärben
plt.axvline(trennindex - 0.5, color='black', linestyle='--', linewidth=1)  # Trennlinie einzeichnen
plt.title('Periodische Zeitreihe mit chronologischem Train/Test-Split')  # Titel setzen
plt.xlabel('Zeitindex')  # x-Achse beschriften
plt.ylabel('Messwert')   # y-Achse beschriften
plt.legend(loc='upper right')  # Legende anzeigen
plt.show()  # Abbildung ausgeben

## Ein kleines Sequence-to-One-RNN trainieren

`SimpleRNN(8)` erzeugt für jeden der 20 Schritte einen Hidden State mit acht Werten. Der letzte Hidden State fasst das gesamte Fenster zusammen. Ein Dense-Layer wandelt ihn in die Ein-Schritt-Prognose um.

Die RNN-Schicht hat `return_sequences=True`. Dadurch können wir später die komplette Folge der Hidden States eines Fensters auslesen.

In [ ]:
# Ein Beispiel besteht aus 20 Zeitschritten mit jeweils einem Messwert.
eingabe = tf.keras.Input(shape=(FENSTERLAENGE, 1), name='zeitfenster')
# Acht RNN-Einheiten erzeugen nach jedem Schritt einen Hidden-State-Vektor mit acht Zahlen.
rnn_schicht = tf.keras.layers.SimpleRNN(8, activation='tanh', return_sequences=True, name='rnn')
# TensorFlow führt intern die Schleife über die 20 Werte aus und gibt alle Hidden States zurück.
hidden_folge = rnn_schicht(eingabe)
# Aus der Folge wird nur der letzte State h(20) für die Ein-Schritt-Prognose ausgewählt.
letzter_hidden_state = tf.keras.layers.Lambda(lambda h: h[:, -1, :], name='letzter_hidden_state')(hidden_folge)
# Ein Dense-Layer bildet den 8-dimensionalen letzten State auf einen vorhergesagten Messwert ab.
prognose = tf.keras.layers.Dense(1, name='naechster_wert')(letzter_hidden_state)
# Dieses Modell wird für Training und Ein-Schritt-Prognosen verwendet.
modell = tf.keras.Model(eingabe, prognose, name='rnn_ein_schritt_prognose')
# Dieses zweite Modell teilt dieselben Gewichte, gibt aber alle Hidden States zurück.
hidden_modell = tf.keras.Model(eingabe, hidden_folge, name='rnn_hidden_states')

# Adam passt die Gewichte an; MSE ist der Trainingsfehler, MAE eine leichter lesbare Zusatzmetrik.
modell.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
# Das Training erhält ausschließlich die frühen Zeitfenster samt ihrer Zielwerte.
historie = modell.fit(
    X_train, y_train,           # Eingabefenster und zugehörige nächste Werte
    validation_split=0.15,      # Die letzten 15 % des Trainingsbereichs dienen zur Validierung
    shuffle=False,              # Reihenfolge bleibt chronologisch
    epochs=45,                  # Anzahl vollständiger Durchläufe über die Trainingsfenster
    batch_size=32,              # 32 Fenster werden pro Gewichtsupdate zusammen verarbeitet
    verbose=0,                  # Keine Textausgabe für jede einzelne Epoche
)

modell.summary()  # Architektur und Tensorformen ausgeben
print()           # Leerzeile zur besseren Lesbarkeit
print(f'Trainierbare Parameter: {modell.count_params():,}')  # Anzahl aller lernbaren Gewichte

## Vorhersagen im späteren Testbereich

Die orange Linie zeigt die aus den Testfenstern entstehenden Ein-Schritt-Prognosen. Jeder Prognosepunkt verwendet ein eigenes 20er-Fenster; der Hidden State startet dafür bei null und wird innerhalb des Fensters bis zum letzten Schritt fortgeführt.

In [ ]:
# Für jedes spätere Testfenster berechnet das RNN eine skalierte Ein-Schritt-Prognose.
test_prognosen_skaliert = modell.predict(X_test, verbose=0).ravel()
# Die Prognosen werden wieder in die ursprüngliche Einheit der Zeitreihe zurückskaliert.
test_prognosen = test_prognosen_skaliert * standardabweichung + mittelwert
# Auch die Testziele werden für einen verständlichen Vergleich zurückskaliert.
test_werte = y_test * standardabweichung + mittelwert
# Der MAE misst den mittleren absoluten Abstand zwischen Prognose und tatsächlichem Zielwert.
test_mae = np.mean(np.abs(test_werte - test_prognosen))

# Die Abbildung besteht aus einem Loss-Verlauf und einem Prognosevergleich.
fig, (ax_loss, ax_prognose) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [1, 2]})
ax_loss.plot(historie.history['loss'], label='Training-Loss')       # Fehler auf den Trainingsfenstern
ax_loss.plot(historie.history['val_loss'], label='Validierung-Loss')  # Fehler auf dem späteren Validierungsabschnitt
ax_loss.set(title='Training des RNN', xlabel='Epoche', ylabel='MSE')  # Achsen und Titel setzen
ax_loss.legend()  # Legende für beide Fehlerkurven anzeigen

ax_prognose.plot(test_indizes, test_werte, color='tab:green', linewidth=2, label='tatsächliche Testwerte')  # Wahre Zielwerte
ax_prognose.plot(test_indizes, test_prognosen, color='tab:orange', linestyle='--', linewidth=1.8, label='RNN-Prognosen')  # Modellwerte
ax_prognose.set(title=f'Ein-Schritt-Prognosen im Testbereich | Test-MAE: {test_mae:.3f}', xlabel='Zeitindex', ylabel='Messwert')  # Beschriftung
ax_prognose.legend(loc='upper right')  # Legende in der oberen rechten Ecke
plt.tight_layout()  # Abstände automatisch anpassen
plt.show()          # Die fertige Abbildung ausgeben

## Hidden State innerhalb eines Fensters

Ein ausgewähltes Testfenster wird jetzt einzeln durch die trainierte RNN-Zelle geführt. Die Heatmap zeigt für jede der acht Hidden-State-Komponenten, wie ihr Wert über die 20 Eingabeschritte fortgeschrieben wird. Der letzte Spaltenvektor ist die Darstellung, die der Dense-Layer für die Vorhersage verwendet.

In [ ]:
BEISPIEL_NUMMER = 12  # Welches Fenster aus dem Testbereich detailliert dargestellt wird
beispiel_fenster = X_test[BEISPIEL_NUMMER : BEISPIEL_NUMMER + 1]  # Batch mit genau einem Fenster
beispiel_werte = beispiel_fenster[0, :, 0] * standardabweichung + mittelwert  # Fenster zurückskalieren
beispiel_ziel = test_werte[BEISPIEL_NUMMER]            # Tatsächlicher 21. Wert
beispiel_prognose = test_prognosen[BEISPIEL_NUMMER]    # Vom RNN vorhergesagter 21. Wert
hidden_states = hidden_modell.predict(beispiel_fenster, verbose=0)[0]  # Alle 20 States mit je 8 Werten

fig, (ax_fenster, ax_hidden) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [1, 2]})  # Zwei Diagrammbereiche
positionen = np.arange(1, FENSTERLAENGE + 1)  # Positionen 1 bis 20 für die x-Achse
ax_fenster.plot(positionen, beispiel_werte, 'o-', color='tab:blue', linewidth=2, label='20 Eingabewerte')  # RNN-Eingabe
ax_fenster.scatter(FENSTERLAENGE + 1, beispiel_ziel, s=95, color='tab:green', zorder=3, label='tatsächlicher nächster Wert')  # Richtiges Ziel
ax_fenster.scatter(FENSTERLAENGE + 1, beispiel_prognose, s=95, color='tab:orange', zorder=3, label='RNN-Prognose')  # Modellprognose
ax_fenster.axvline(FENSTERLAENGE + 0.5, color='0.4', linestyle=':')  # Trennung zwischen Eingabe und Ziel
ax_fenster.set(title='Ein Testfenster und die Vorhersage für den nächsten Wert', xlabel='Position im Fenster', ylabel='Messwert')  # Beschriftung
ax_fenster.legend(loc='upper left')  # Legende anzeigen

bild = ax_hidden.imshow(hidden_states.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)  # States als Heatmap: Zeilen=Einheiten, Spalten=Schritte
ax_hidden.set(title='Hidden States des gleichen Fensters', xlabel='Zeitschritt im Fenster', ylabel='Hidden-State-Komponente')  # Beschriftung
ax_hidden.set_yticks(np.arange(8), [f'h[{i}]' for i in range(8)])  # Namen für die acht Einheiten
fig.colorbar(bild, ax=ax_hidden, label='Aktivierung')  # Farbskala der Heatmap
plt.tight_layout()  # Abstände anpassen
plt.show()          # Abbildung ausgeben

print('Form des Hidden-State-Tensors:', hidden_states.shape, '= (20 Zeitschritte, 8 Hidden-State-Werte)')  # Tensorform erklären
print('Letzter Hidden State:', np.round(hidden_states[-1], 3))  # Dieser Vektor geht an den Dense-Layer

## Der RNN-Durchlauf als expliziter `for`-Loop

`SimpleRNN` führt intern denselben Ablauf aus: Es nimmt einen Wert, kombiniert ihn mit dem bisherigen Hidden State und speichert den neuen State für den nächsten Schritt. Die folgende Schleife verwendet die trainierte TensorFlow-RNN-Zelle direkt und reproduziert den Hidden-State-Verlauf aus der Heatmap.

In [ ]:
hidden_state = tf.zeros((1, rnn_schicht.units), dtype=tf.float32)  # Für dieses neue Fenster startet h(0) als Nullvektor
manuelle_hidden_states = []  # Hier speichern wir den neuen State nach jedem Schleifendurchlauf

for x_t in beispiel_fenster[0]:  # Die 20 Eingabewerte werden tatsächlich einzeln nacheinander verarbeitet
    ausgabe, [hidden_state] = rnn_schicht.cell(x_t.reshape(1, 1), states=[hidden_state], training=False)  # x(t) und h(t-1) erzeugen h(t)
    manuelle_hidden_states.append(hidden_state.numpy()[0])  # Den gerade entstandenen h(t)-Vektor merken

manuelle_hidden_states = np.asarray(manuelle_hidden_states)  # Python-Liste in ein NumPy-Array umwandeln
np.testing.assert_allclose(manuelle_hidden_states, hidden_states, atol=1e-6)  # Prüfen: Loop und Keras liefern denselben Verlauf

print('Schritt | Eingabe x(t) | Norm des neuen Hidden State')  # Tabellenkopf ausgeben
print('-' * 52)  # Trennlinie ausgeben
for schritt, (x_t, h_t) in enumerate(zip(beispiel_werte, manuelle_hidden_states), start=1):  # Jeden Schritt lesbar ausgeben
    print(f'{schritt:>7} | {x_t:>12.3f} | {np.linalg.norm(h_t):>30.3f}')  # Eingabe und Größe des h(t)-Vektors

print('\nNach Schritt 20 wird der letzte Hidden State an den Dense-Layer übergeben.')  # Übergang von RNN zu Prognosekopf

## Ein fortlaufender Durchlauf mit passendem und unpassendem Hidden State

Für unabhängige 20er-Fenster beginnt der Hidden State immer wieder bei null. Bei einer fortlaufend gemessenen Zeitreihe kann derselbe RNN-Kern dagegen über die gesamte Sequenz laufen. Der Hidden State des vorherigen Zeitpunkts wird dann direkt für den nächsten Zeitpunkt weiterverwendet.

Der Vergleich startet denselben Testabschnitt einmal mit einem State aus seiner tatsächlichen Vorgeschichte und einmal mit einem State aus einer gegensätzlichen Phase der Zeitreihe. Der unpassende State beeinflusst die ersten Prognosen, bevor die neuen Eingabewerte ihn schrittweise überschreiben.

In [ ]:
# Wir betrachten einen kurzen, späteren Ausschnitt der Testzeitreihe.
SEGMENT_START = trennindex + 25  # Startposition des zu prognostizierenden Testabschnitts
SEGMENT_LAENGE = 45             # Anzahl fortlaufend zu verarbeitender Messwerte
SEGMENT_ENDE = SEGMENT_START + SEGMENT_LAENGE  # Exklusive Endposition des Ausschnitts

# Die echte Vorgeschichte sind die 20 Werte unmittelbar vor dem neuen Abschnitt.
passende_vorgeschichte = werte_skaliert[SEGMENT_START - FENSTERLAENGE : SEGMENT_START]
# Eine um 15 Schritte verschobene Vorgeschichte liegt bei diesem Signal ungefähr in der Gegenphase.
falsche_vorgeschichte = werte_skaliert[SEGMENT_START - FENSTERLAENGE - 15 : SEGMENT_START - 15]
# Der passende Start-State entsteht durch Verarbeitung der tatsächlichen Vorgeschichte.
passender_start_state = hidden_modell.predict(passende_vorgeschichte.reshape(1, -1, 1), verbose=0)[0, -1]
# Der unpassende Start-State stammt aus einer anderen Phase der Zeitreihe.
falscher_start_state = hidden_modell.predict(falsche_vorgeschichte.reshape(1, -1, 1), verbose=0)[0, -1]
# Dieser Tensor enthält die neuen Messwerte, die in beiden Durchläufen identisch sind.
segment = werte_skaliert[SEGMENT_START : SEGMENT_ENDE].reshape(1, -1, 1).astype(np.float32)

# Die RNN-Schicht verarbeitet das Segment mit dem State aus der richtigen Vorgeschichte weiter.
states_passend = rnn_schicht(segment, initial_state=[tf.convert_to_tensor(passender_start_state.reshape(1, -1), dtype=tf.float32)], training=False)
# Die gleiche Eingabe startet diesmal absichtlich mit einem State aus der falschen Phase.
states_falsch = rnn_schicht(segment, initial_state=[tf.convert_to_tensor(falscher_start_state.reshape(1, -1), dtype=tf.float32)], training=False)
# Der bereits trainierte Dense-Layer macht aus jedem Hidden State eine Vorhersage für den nächsten Wert.
vorhersage_layer = modell.get_layer('naechster_wert')
prognosen_passend = vorhersage_layer(states_passend).numpy()[0, :, 0] * standardabweichung + mittelwert
prognosen_falsch = vorhersage_layer(states_falsch).numpy()[0, :, 0] * standardabweichung + mittelwert
# TensorFlow-Tensoren werden für die spätere Darstellung in NumPy-Arrays umgewandelt.
states_passend = states_passend.numpy()[0]
states_falsch = states_falsch.numpy()[0]

# Eine Prognose nach x(t) wird mit dem tatsächlich folgenden Wert x(t+1) verglichen.
vergleichszeit = zeit[SEGMENT_START + 1 : SEGMENT_ENDE]
vergleichsziele = werte[SEGMENT_START + 1 : SEGMENT_ENDE]
mae_passend = np.mean(np.abs(vergleichsziele - prognosen_passend[:-1]))
mae_falsch_erste_10 = np.mean(np.abs(vergleichsziele[:10] - prognosen_falsch[:10]))

# Oben stehen die Vorhersagen, unten die Distanz zwischen beiden Hidden-State-Verläufen.
fig, (ax_prognose, ax_abstand) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)  # Diagramm für Prognosen und State-Abstand
ax_prognose.plot(vergleichszeit, vergleichsziele, color='tab:green', linewidth=2, label='tatsächlicher nächster Wert')  # Wahre Folgewertkurve
ax_prognose.plot(vergleichszeit, prognosen_passend[:-1], color='tab:blue', linewidth=1.8, label='Prognose mit passendem Start-State')  # Referenzprognose
ax_prognose.plot(vergleichszeit, prognosen_falsch[:-1], color='crimson', linestyle='--', linewidth=1.8, label='Prognose mit unpassendem Start-State')  # Von falscher Erinnerung beeinflusste Prognose
ax_prognose.axvspan(vergleichszeit[0], vergleichszeit[9], color='crimson', alpha=0.08, label='erste 10 beeinflusste Schritte')  # Anfangsbereich hervorheben
ax_prognose.set(title=f'Ein falscher Hidden State stört besonders den Beginn des Abschnitts | MAE passend: {mae_passend:.3f}, falsch in ersten 10 Schritten: {mae_falsch_erste_10:.3f}', ylabel='Messwert')  # Titel und Achse
ax_prognose.legend(loc='upper right')  # Legende anzeigen

# Die euklidische Distanz zeigt, wie stark sich passender und falscher State noch unterscheiden.
state_abstand = np.linalg.norm(states_passend - states_falsch, axis=1)  # Abstand der beiden 8-dimensionalen State-Vektoren berechnen
ax_abstand.plot(zeit[SEGMENT_START : SEGMENT_ENDE], state_abstand, color='black', linewidth=2, label='Abstand der Hidden States')  # Distanzverlauf zeichnen
ax_abstand.axhline(0, color='0.5', linewidth=0.8)  # Null-Linie als Referenz
ax_abstand.set(title='Die neuen Eingaben überschreiben den unpassenden Start-State schrittweise', xlabel='Zeitindex', ylabel='Distanz')  # Achsen beschriften
ax_abstand.legend(loc='upper right')  # Legende anzeigen
plt.tight_layout()  # Abstände zwischen beiden Diagrammen anpassen
plt.show()          # Abbildung ausgeben

print('Passender Start-State:', np.round(passender_start_state, 3))  # State aus der richtigen Vorgeschichte zeigen
print('Unpassender Start-State:', np.round(falscher_start_state, 3))  # State aus der Gegenphase zeigen
print(f'MAE mit passendem State über den Abschnitt: {mae_passend:.3f}')  # Fehler der Referenzprognose ausgeben
print(f'MAE mit unpassendem State in den ersten 10 Schritten: {mae_falsch_erste_10:.3f}')  # Anfangsfehler des falschen States ausgeben

## Kerngedanke

Das RNN verarbeitet ein Fenster schrittweise und aktualisiert dabei seinen Hidden State. Der letzte State liefert die Information für die Ein-Schritt-Prognose. Bei einzelnen Trainingsfenstern beginnt der State neu; bei einem fortlaufenden Durchlauf wird er über die gesamte Zeitreihe weitergegeben.